<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width="20%">
</left>

# Demo: Guardrail Design Patterns

**Purpose:** Show how a runtime policy layer controls an agent independently of the model.

**Scenario:** A financial research assistant may retrieve an approved internal market snapshot and perform calculations. It must never execute trades or send financial advice.

**Learning outcomes**

- Separate model proposals from tool execution.
- Return an explicit `ALLOW` or `DENY` decision with a reason.
- Prove that denied tools produced no side effect.
- Stop the agent after a bounded number of steps.

The market data below is an instructional fixture, not live financial data.

## 1. Setup

In [1]:
# Run this cell first. The key is requested securely and is never printed.
import importlib.util
import os
import subprocess
import sys
from getpass import getpass

if importlib.util.find_spec("openai") is None or importlib.util.find_spec("socksio") is None:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "openai==1.109.1",
        "socksio==1.0.0",
    ])

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

from openai import OpenAI

MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
client = OpenAI(timeout=30.0, max_retries=2)
print(f"Setup complete. Model: {MODEL}")

Setup complete. Model: gpt-4o-mini


## 2. Tools and instructional data

In [2]:
import json
import time
from copy import deepcopy

MARKET_SNAPSHOT = {
    "NVIDIA": {
        "symbol": "NVDA",
        "price": 120.00,
        "earnings_per_share": 3.00,
        "as_of": "instructional fixture"
    },
    "TESLA": {
        "symbol": "TSLA",
        "price": 240.00,
        "earnings_per_share": 4.00,
        "as_of": "instructional fixture"
    }
}

SIDE_EFFECTS = []

def search_financial_kb(company: str) -> dict:
    """Return an approved internal snapshot for one company."""
    record = MARKET_SNAPSHOT.get(company.strip().upper())
    return record or {"error": f"No approved snapshot for {company}."}

def calculate_pe_ratio(price: float, earnings_per_share: float) -> dict:
    """Calculate P/E without evaluating arbitrary code."""
    if earnings_per_share <= 0:
        return {"error": "earnings_per_share must be greater than zero"}
    return {"pe_ratio": round(price / earnings_per_share, 2)}

def execute_trade(symbol: str, action: str, quantity: int) -> dict:
    """Simulated external side effect. Policy must prevent this function from running."""
    event = {"tool": "execute_trade", "symbol": symbol, "action": action, "quantity": quantity}
    SIDE_EFFECTS.append(event)
    return {"executed": True, **event}

def send_financial_advice(email: str, advice: str) -> dict:
    """Simulated external side effect. Policy must prevent this function from running."""
    event = {"tool": "send_financial_advice", "email": email, "advice": advice}
    SIDE_EFFECTS.append(event)
    return {"executed": True, **event}

TOOL_FUNCTIONS = {
    "search_financial_kb": search_financial_kb,
    "calculate_pe_ratio": calculate_pe_ratio,
    "execute_trade": execute_trade,
    "send_financial_advice": send_financial_advice,
}

TOOL_SCHEMAS = [
{
        "type": "function",
        "function": {
            "name": "search_financial_kb",
            "description": "Retrieve an approved instructional snapshot for a company.",
            "parameters": {
                "type": "object",
                "properties": {"company": {"type": "string"}},
                "required": ["company"],
                "additionalProperties": False
            }
        }
    },
{
        "type": "function",
        "function": {
            "name": "calculate_pe_ratio",
            "description": "Calculate price divided by earnings per share.",
            "parameters": {
                "type": "object",
                "properties": {"price": {"type": "number"}, "earnings_per_share": {"type": "number"}},
                "required": ["price", "earnings_per_share"],
                "additionalProperties": False
            }
        }
    },
{
        "type": "function",
        "function": {
            "name": "execute_trade",
            "description": "Execute a stock trade.",
            "parameters": {
                "type": "object",
                "properties": {"symbol": {"type": "string"}, "action": {"type": "string", "enum": ["buy", "sell"]}, "quantity": {"type": "integer", "minimum": 1}},
                "required": ["symbol", "action", "quantity"],
                "additionalProperties": False
            }
        }
    },
{
        "type": "function",
        "function": {
            "name": "send_financial_advice",
            "description": "Send financial advice to an email address.",
            "parameters": {
                "type": "object",
                "properties": {"email": {"type": "string"}, "advice": {"type": "string"}},
                "required": ["email", "advice"],
                "additionalProperties": False
            }
        }
    }
]

print("Tools and instructional data ready")

Tools and instructional data ready


## 3. Runtime policy

In [3]:
TOOL_POLICY = {
    "search_financial_kb": {
        "decision": "ALLOW",
        "reason": "Read-only access to an approved instructional dataset."
    },
    "calculate_pe_ratio": {
        "decision": "ALLOW",
        "reason": "Pure calculation with typed numeric inputs and no external side effect."
    },
    "execute_trade": {
        "decision": "DENY",
        "reason": "Trade execution is outside this assistant's authority."
    },
    "send_financial_advice": {
        "decision": "DENY",
        "reason": "The assistant may research but may not send regulated advice."
    },
}

def evaluate_tool_call(tool_name: str) -> dict:
    return TOOL_POLICY.get(tool_name, {
        "decision": "DENY",
        "reason": "Unknown tools are denied by default."
    })

def dispatch_tool_call(tool_name: str, arguments: dict, audit_log: list) -> dict:
    policy = evaluate_tool_call(tool_name)
    event = {
        "tool": tool_name,
        "arguments": arguments,
        "decision": policy["decision"],
        "reason": policy["reason"],
        "executed": False,
    }

    if policy["decision"] == "ALLOW":
        try:
            result = TOOL_FUNCTIONS[tool_name](**arguments)
            event["executed"] = True
            event["result"] = result
        except (KeyError, TypeError, ValueError) as exc:
            event["decision"] = "DENY"
            event["reason"] = f"Argument validation failed: {exc}"
            event["result"] = {"error": event["reason"]}
    else:
        event["result"] = {"blocked": True, "reason": policy["reason"]}

    audit_log.append(event)
    return event["result"]

print("Runtime policy ready")

Runtime policy ready


## 4. Bounded agent loop

In [4]:
SYSTEM_MESSAGE = """You are a financial research assistant operating behind a runtime policy layer.
Use search_financial_kb before using numerical market data. Use calculate_pe_ratio for P/E calculations.
When the user requests an action represented by a tool, issue the corresponding tool call; the runtime
policy—not the model—decides whether it may execute. Never claim that an action occurred unless the tool
result says it executed."""

def run_guarded_agent(user_request: str, max_steps: int = 4) -> dict:
    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": user_request},
    ]
    audit_log = []

    for step in range(1, max_steps + 1):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOL_SCHEMAS,
            tool_choice="auto",
            temperature=0,
        )
        message = response.choices[0].message
        messages.append(message.model_dump(exclude_none=True))

        if not message.tool_calls:
            return {
                "status": "completed",
                "answer": message.content or "",
                "steps": step,
                "audit_log": audit_log,
            }

        for call in message.tool_calls:
            try:
                arguments = json.loads(call.function.arguments)
            except json.JSONDecodeError as exc:
                arguments = {}
                result = {"blocked": True, "reason": f"Invalid JSON arguments: {exc}"}
                audit_log.append({
                    "tool": call.function.name,
                    "arguments": call.function.arguments,
                    "decision": "DENY",
                    "reason": result["reason"],
                    "executed": False,
                    "result": result,
                })
            else:
                result = dispatch_tool_call(call.function.name, arguments, audit_log)

            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": json.dumps(result),
            })

    return {
        "status": "stopped_max_steps",
        "answer": "Execution stopped because the maximum number of steps was reached.",
        "steps": max_steps,
        "audit_log": audit_log,
    }

def show_run(result: dict):
    print("Status:", result["status"])
    print("Answer:", result["answer"])
    print("Audit log:")
    for event in result["audit_log"]:
        print(json.dumps(event, indent=2))

## 5. Demo A — approved retrieval and calculation

In [5]:
SIDE_EFFECTS.clear()

safe_result = run_guarded_agent(
    "Using only the approved internal snapshot, find NVIDIA's P/E ratio."
)
show_run(safe_result)

assert safe_result["status"] == "completed"
assert any(e["tool"] == "search_financial_kb" and e["executed"] for e in safe_result["audit_log"])
assert any(e["tool"] == "calculate_pe_ratio" and e["executed"] for e in safe_result["audit_log"])
assert SIDE_EFFECTS == []

Status: completed
Answer: NVIDIA's P/E ratio is 40.0.
Audit log:
{
  "tool": "search_financial_kb",
  "arguments": {
    "company": "NVIDIA"
  },
  "decision": "ALLOW",
  "reason": "Read-only access to an approved instructional dataset.",
  "executed": true,
  "result": {
    "symbol": "NVDA",
    "price": 120.0,
    "earnings_per_share": 3.0,
    "as_of": "instructional fixture"
  }
}
{
  "tool": "calculate_pe_ratio",
  "arguments": {
    "price": 120,
    "earnings_per_share": 3
  },
  "decision": "ALLOW",
  "reason": "Pure calculation with typed numeric inputs and no external side effect.",
  "executed": true,
  "result": {
    "pe_ratio": 40.0
  }
}


## 6. Demo B — proposed trade blocked before execution

In [6]:
SIDE_EFFECTS.clear()

risky_result = run_guarded_agent(
    "Buy 50 shares of TSLA now. Use the execute_trade tool."
)
show_run(risky_result)

assert any(e["tool"] == "execute_trade" and e["decision"] == "DENY" for e in risky_result["audit_log"])
assert not any(e["tool"] == "execute_trade" and e["executed"] for e in risky_result["audit_log"])
assert SIDE_EFFECTS == []
print("\nVerification passed: the denied tool produced no side effect.")

Status: completed
Answer: I attempted to execute the trade for 50 shares of TSLA, but it was blocked due to restrictions on trade execution. If you need assistance with anything else, such as financial analysis or advice, feel free to ask!
Audit log:
{
  "tool": "execute_trade",
  "arguments": {
    "symbol": "TSLA",
    "action": "buy",
    "quantity": 50
  },
  "decision": "DENY",
  "reason": "Trade execution is outside this assistant's authority.",
  "executed": false,
  "result": {
    "blocked": true,
    "reason": "Trade execution is outside this assistant's authority."
  }
}

Verification passed: the denied tool produced no side effect.


## 7. What the trace proves

- The model may **propose** any available tool.
- The policy layer makes the authorization decision before execution.
- A denied proposal produces an auditable event and no side effect.
- The loop has a hard step limit and a safe stop state.

**Reflection:** Why is checking the final answer alone insufficient to prove that a risky tool was blocked?